Chapter 5 covers Text Analysis, Unstructured Data Parsing, and Regular Expressions.

In real-world analytical and quantitative pipelines, raw incoming logs, transaction descriptions, policy details, and web data are rarely neatly categorized into clean numeric columns. Chapter 5 focuses on how to parse, extract, clean, and categorize unstructured text directly within the database engine before feeding downstream tables into analytical models.

The 4 Core Takeaways of Chapter 5

1. Structural Text Extraction vs. Full Search
   There are two primary reasons to process text in SQL:

Parsing / Extraction: Pulling standardized sub-components out of structured text strings (e.g., extracting domain names from email addresses, pulling transaction reference codes from bank logs, or parsing URL query parameters).

Classification / Categorization: Mapping messy, high-cardinality free-text descriptions into clean, low-cardinality categorical variables using conditional string matching (LIKE, ILIKE, or regex pattern matches).

2. Standard String Manipulation Functions
   For simple, predictable string structures with consistent delimiters, standard SQL string functions are faster and less compute-heavy than regular expression engines.

Length & Trimming: LENGTH(), TRIM(), LTRIM(), RTRIM().

Slicing & Substrings: SUBSTRING(string FROM start FOR length), LEFT(string, n), RIGHT(string, n).

Delimiter Splitting: SPLIT_PART(string, delimiter, index) (e.g., SPLIT_PART('user@domain.com', '@', 2) extracts 'domain.com').

Replacement: REPLACE(string, search_term, replace_term).

3. Regular Expressions (REGEXP / RLIKE)
   When string structures vary or follow complex patterns, regular expressions are required. DuckDB and PostgreSQL support POSIX regex operators and functions natively.

Pattern Matching (regexp_matches / ~): Filtering rows based on regex patterns (e.g., validating formatted SSNs, zip codes, or account numbers).

Extraction (regexp_extract): Capturing specific wildcard groups or numeric tokens from mixed text fields.

Example: REGEXP_EXTRACT(description, 'INV-([0-9]+)', 1) extracts the numeric invoice ID from 'Payment for INV-984321 on account'.

Replacement (regexp_replace): Stripping non-numeric characters from dirty numeric strings (e.g., removing spaces, commas, and currency symbols simultaneously).

4. Text Similarity and Distance Metrics
   When attempting to match fuzzy entities (e.g., matching company names like "Acme Corp" and "Acme Corporation" across separate data sources), SQL engines provide string distance algorithms:

Levenshtein Distance: Measures the minimum number of single-character edits (insertions, deletions, substitutions) required to change one string into another.

Jaro-Winkler / Soundex: Phonetic and structural matching algorithms for fuzzy name resolution.


In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=":memory:")

# 1. Create a mock dataset containing messy financial transaction descriptions
conn.execute("""
CREATE TABLE raw_banking_logs (
    tx_id INT,
    raw_description VARCHAR,
    raw_amount VARCHAR
);

INSERT INTO raw_banking_logs VALUES
    (101, 'POS Purchase - AMZN MKTP US*2X49A1 Seattle WA', '$1,250.00'),
    (102, 'WIRE TRANS REF: W-908234 FROM CHASE BANK', '$50,000.50'),
    (103, 'PAYROLL DIRECT DEP - ACME CORP ID: 44102', '$3,400.00'),
    (104, 'POS Purchase - Starbucks Store #04921 Irvine CA', '$6.75'),
    (105, 'ATM CASH WITHDRAWAL - BANK OF AMERICA #8812', '$200.00');
""")

# 2. Chapter 5 Text Analysis & Regex Extraction Pipeline
text_parsed_df = conn.execute("""
SELECT
    tx_id,
    raw_description,

    -- Pattern 1: Categorization via Case-Insensitive Pattern Matching
    CASE
        WHEN raw_description ILIKE '%POS Purchase%' THEN 'Merchant POS'
        WHEN raw_description ILIKE '%WIRE TRANS%' THEN 'Wire Transfer'
        WHEN raw_description ILIKE '%PAYROLL%' THEN 'Payroll'
        WHEN raw_description ILIKE '%ATM CASH%' THEN 'ATM Withdrawal'
        ELSE 'Other'
    END AS tx_category,

    -- Pattern 2: Regex Extraction of Reference/Store Numbers
    REGEXP_EXTRACT(raw_description, '(REF: [A-Z0-9-]+|Store #[0-9]+|ID: [0-9]+)', 1) AS extracted_ref_id,

    -- Pattern 3: Standard String Splitting (Pulling last word / state code)
    SPLIT_PART(TRIM(raw_description), ' ', -1) AS trailing_token,

    -- Pattern 4: Regex Cleaning & Numeric Casting of Currency Strings
    CAST(REGEXP_REPLACE(raw_amount, '[$,]', '', 'g') AS NUMERIC(10,2)) AS clean_amount
FROM raw_banking_logs;
""").df()

text_parsed_df

| Concept                  | Problem Solved                                               | Primary SQL Function / Syntax                  |
| :----------------------- | :----------------------------------------------------------- | :--------------------------------------------- |
| **Substring Extraction** | Pulling known slices out of delimited strings                | `SPLIT_PART(str, delim, pos)`, `SUBSTRING()`   |
| **Regex Capture**        | Extracting dynamic variable tokens from unstructured text    | `REGEXP_EXTRACT(str, 'pattern', group_idx)`    |
| **Regex Cleanup**        | Bulk stripping illegal/non-numeric characters                | `REGEXP_REPLACE(str, '[^0-9.]', '', 'g')`      |
| **Category Mapping**     | Grouping free-text descriptions into low-cardinality buckets | `CASE WHEN str ILIKE '%pattern%' THEN ... END` |
| **Fuzzy Matching**       | Measuring character distance between unstandardized names    | `LEVENSHTEIN(str1, str2)`                      |
